# SQL Query Modernization with Mellea

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibm-granite-community/mellea-cookbook/blob/main/recipes/SQLModernization/SQLModernization.ipynb)

This recipe modernizes a legacy SQL query with Mellea's **Instruct-Validate-Repair (IVR)** pattern. The model rewrites a query that hardcodes a player name, and every rewrite must pass a set of requirements before it is accepted. Failed attempts are repaired with feedback until they pass or the loop budget runs out.

## Scenario

A cricket statistics dashboard uses a legacy query that hardcodes one player, Jasprit Bumrah. We want a version that:

1. Adds match-level columns from a joined table (`match_details`).
2. Takes the player name as a parameter (`:player_name`), so one query serves every player.
3. Keeps every column the dashboard already reads.

We'll build an in-memory SQLite database with three players, run the legacy query, ask Mellea for the modernized query, and then check its results against ground truth for each player.

## What You'll Build

- An in-memory SQLite database with cricket statistics for three players.
- A modernized query that adds match-level columns and takes the player name as a `:player_name` parameter.
- Results for each player, checked against the source table.

## Prerequisites

- Python 3.11 or later.
- [Ollama](https://ollama.com/) running locally with the `granite4.1:3b` model. No API keys are needed.

## Step 1. Set up your environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it and run locally. We recommend a [virtual environment](https://docs.python.org/3/library/venv.html) to avoid dependency conflicts.

In [ ]:
# Uncomment to install dependencies in Colab or a fresh environment
# !pip install mellea sqlparse pandas

### Set up a backend

This recipe uses a local Ollama server with the IBM Granite model, which needs no credentials. To use watsonx.ai or another backend instead, skip the cell below and change the `start_session` call in Step 6.

In [ ]:
import shutil
import subprocess
import time
import urllib.request

# Install Ollama only if it is missing (Colab/Linux); on macOS install it yourself with `brew install ollama`.
if shutil.which("ollama") is None:
    !curl -fsSL https://ollama.com/install.sh | sh

def ollama_is_up(url="http://127.0.0.1:11434/api/tags") -> bool:
    try:
        urllib.request.urlopen(url, timeout=1)
        return True
    except Exception:
        return False

if not ollama_is_up():
    # Detached so the server outlives this cell
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(30):
        if ollama_is_up():
            break
        time.sleep(1)

print("Ollama running:", ollama_is_up())


In [ ]:
!ollama pull granite4.1:3b

In [ ]:
import re
import sqlite3
from typing import Literal

import pandas as pd
import pydantic
import sqlparse
from mellea import generative, start_session
from mellea.stdlib.requirement import req, simple_validate
from mellea.stdlib.sampling import RepairTemplateStrategy

## Step 2. Create the database

We'll create two tables. `match_details` has one row per `player_stats` row, joined on `player_id`.

In [ ]:
conn = sqlite3.connect(":memory:", check_same_thread=False)

SCHEMA_DDL = """
CREATE TABLE player_stats (
    player_id INTEGER PRIMARY KEY,
    player_name TEXT NOT NULL,
    year INTEGER,
    total_matches INTEGER,
    wins INTEGER,
    losses INTEGER,
    win_percentage REAL
);
CREATE TABLE match_details (
    detail_id INTEGER PRIMARY KEY,
    player_id INTEGER REFERENCES player_stats(player_id),
    fastest_delivery_speed REAL,
    most_points_won_against TEXT,
    most_points_lost_against TEXT
);
"""
conn.executescript(SCHEMA_DDL)

player_stats = [
    (1, "Jasprit Bumrah", 2023, 28, 19, 9, 67.9),
    (2, "Jasprit Bumrah", 2022, 15, 10, 5, 66.7),
    (3, "Jasprit Bumrah", 2021, 24, 16, 8, 66.7),
    (4, "Mohammed Shami", 2023, 22, 13, 9, 59.1),
    (5, "Mohammed Shami", 2022, 18, 10, 8, 55.6),
    (6, "Ravindra Jadeja", 2023, 30, 20, 10, 66.7),
]
match_details = [
    (1, 1, 153.0, "Australia", "England"),
    (2, 2, 150.6, "South Africa", "Australia"),
    (3, 3, 152.1, "England", "Pakistan"),
    (4, 4, 147.8, "Pakistan", "India"),
    (5, 5, 146.2, "India", "Australia"),
    (6, 6, 121.5, "New Zealand", "England"),
]
conn.executemany("INSERT INTO player_stats VALUES (?, ?, ?, ?, ?, ?, ?)", player_stats)
conn.executemany("INSERT INTO match_details VALUES (?, ?, ?, ?, ?)", match_details)
conn.commit()

display(pd.read_sql_query("SELECT * FROM player_stats", conn))

## Step 3. Run the legacy query

This query hardcodes the player name and returns only the original columns.

In [ ]:
ORIGINAL_QUERY = """
SELECT
    player_name,
    total_matches,
    wins,
    losses,
    win_percentage
FROM player_stats
WHERE player_name = 'Jasprit Bumrah'
ORDER BY year DESC;
"""

df_original = pd.read_sql_query(ORIGINAL_QUERY, conn)
display(df_original)

## Step 4. Define the requirements

Each requirement is a validator that Mellea runs on every candidate query. Three are deterministic, which makes them fast and repeatable. The fourth uses a `@generative` function, so a model judges whether a name is hardcoded.

`clean_sql` removes Markdown code fences, which models often add around SQL.

In [ ]:
REQUIRED_COLUMNS = ["player_name", "total_matches", "wins", "losses", "win_percentage"]
NEW_COLUMNS = ["fastest_delivery_speed", "most_points_won_against", "most_points_lost_against"]


def clean_sql(text: str) -> str:
    """Strip Markdown code fences that models often wrap around SQL."""
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    return (match.group(1) if match else text).strip()


class HardcodedParameterCheck(pydantic.BaseModel):
    contains_hardcoded: Literal["yes", "no"]
    hardcoded_value: str | None = None  # The hardcoded value found, if any


def validate_sql_syntax(sql: str) -> bool:
    """Check that the query is a single SELECT and that SQLite can plan it against the real schema."""
    sql = clean_sql(sql)
    selects = [s for s in sqlparse.parse(sql) if s.get_type() == "SELECT"]
    if len(selects) != 1:
        print("SQL validation failed: expected exactly one SELECT statement")
        return False
    # Give every named parameter a placeholder value so EXPLAIN can run without executing the query
    params = {name: "placeholder" for name in re.findall(r":(\w+)", sql)}
    try:
        conn.execute(f"EXPLAIN QUERY PLAN {sql}", params)
        return True
    except sqlite3.Error as e:
        print(f"SQL validation failed: {e}")
        return False


def validate_backward_compatible(sql: str) -> bool:
    """Check that the new query still returns every original column."""
    sql = clean_sql(sql)
    return all(re.search(rf"\b{col}\b", sql, re.IGNORECASE) for col in REQUIRED_COLUMNS)


def validate_uses_named_parameter(sql: str) -> bool:
    """Check for the :player_name parameter and no quoted string literals."""
    sql = clean_sql(sql)
    return ":player_name" in sql and re.search(r"'[^']*'", sql) is None


@generative
def contains_hardcoded_parameters(sql_query: str) -> HardcodedParameterCheck:
    """Determines if the SQL query contains hardcoded values for parameters such as player names, player ids, dates etc. Instead of specific names, id or dates, the SQL query must use variables such as player_name, player_id, date etc.

    Returns:
        HardcodedParameterCheck with:
        - contains_hardcoded: 'yes' if hardcoded parameters found, 'no' otherwise
        - hardcoded_value: The actual hardcoded value found (e.g., 'Jasprit Bumrah'), or None if no hardcoded values
    """

## Step 5. Start the session and check whether modernization is needed

A `@generative` function returns a typed result, so the decision below is a plain string.

In [ ]:
m = start_session(backend_name="ollama", model_id="granite4.1:3b")


@generative
def needs_modernization_or_contains_hardcoded_parameter(sql_query: str) -> Literal["yes", "no"]:
    """Determines if the SQL query is outdated or has hardcoded parameter. If it has hardcoded parameter it needs modernization. Parameters can be things like player names, player ids, dates etc."""


needs_update = needs_modernization_or_contains_hardcoded_parameter(m, sql_query=ORIGINAL_QUERY)
print(f"Needs modernization? {needs_update}")

## Step 6. Modernize the query with Instruct-Validate-Repair

`m.instruct` samples a query and runs every requirement. If any fails, the failure feedback goes back into the next attempt. `RepairTemplateStrategy(loop_budget=5)` allows up to five attempts.

In [ ]:
def validate_no_hardcoded_params(sql: str) -> bool:
    """Ask the generative check whether the query hardcodes any value."""
    result = contains_hardcoded_parameters(m, sql_query=clean_sql(sql))
    if result.contains_hardcoded == "yes":
        print(f"⚠️  Hardcoded parameter detected: {result.hardcoded_value}")
        return False
    return True


MODERNIZE_INSTRUCTION = """Modernize the SQL query below:
- Add fastest_delivery_speed, most_points_won_against, and most_points_lost_against from the match_details table.
- Join match_details to player_stats on player_id.
- Replace the hardcoded player name with the :player_name parameter.
- Keep every original column.

Return ONLY the SQL query."""

if needs_update == "yes":
    result = m.instruct(
        MODERNIZE_INSTRUCTION,
        grounding_context={"original_query": ORIGINAL_QUERY, "schema": SCHEMA_DDL},
        requirements=[
            req("SQL must be a single valid SELECT statement", validation_fn=simple_validate(validate_sql_syntax)),
            req("SQL must include all original columns", validation_fn=simple_validate(validate_backward_compatible)),
            req("SQL must use the :player_name parameter and no string literals", validation_fn=simple_validate(validate_uses_named_parameter)),
            req("SQL must not hardcode any value (checked with @generative)", validation_fn=simple_validate(validate_no_hardcoded_params)),
        ],
        strategy=RepairTemplateStrategy(loop_budget=5),
        return_sampling_results=True,
    )

    if result.success:
        modernized_query = clean_sql(str(result.result))
        print(f"✓ Passed all requirements after {len(result.sample_generations)} attempt(s)")
        print(f"\nModernized query:\n{modernized_query}")
    else:
        modernized_query = None
        print(f"✗ No passing query after {len(result.sample_generations)} attempt(s)")
else:
    modernized_query = None
    print("Query is already modern")

## Step 7. Run the modernized query for several players

Named parameters work directly with SQLite, so the same query runs for every player with no string substitution.

In [ ]:
PLAYERS = ["Jasprit Bumrah", "Mohammed Shami", "Ravindra Jadeja"]

if modernized_query:
    frames = []
    for name in PLAYERS:
        frame = pd.read_sql_query(modernized_query, conn, params={"player_name": name})
        frames.append(frame)
    modern_df = pd.concat(frames, ignore_index=True)
    display(modern_df)
else:
    modern_df = None
    print("No modernized query to execute")

## Step 8. Validate against ground truth

For each player, we query `player_stats` directly and compare the core columns with the modernized results. We also check that the new columns are filled in.

In [ ]:
CORE_COLUMNS = ["player_name", "total_matches", "wins", "losses", "win_percentage"]

if modern_df is None:
    print("Cannot validate: no modernized query was produced")
else:
    all_passed = True
    for name in PLAYERS:
        expected = pd.read_sql_query(
            "SELECT player_name, total_matches, wins, losses, win_percentage FROM player_stats WHERE player_name = ?",
            conn,
            params=(name,),
        )
        actual = modern_df.loc[modern_df["player_name"] == name, CORE_COLUMNS]
        key = list(expected.columns)
        try:
            pd.testing.assert_frame_equal(
                expected.sort_values(key).reset_index(drop=True),
                actual.sort_values(key).reset_index(drop=True),
            )
            print(f"✓ {name}: {len(expected)} row(s) match ground truth")
        except AssertionError as e:
            all_passed = False
            print(f"✗ {name}: {e}")

    new_cols_filled = modern_df[NEW_COLUMNS].notna().all().all()
    print(f"{'✓' if new_cols_filled else '✗'} New columns are populated")
    all_passed = all_passed and new_cols_filled
    print("\nAll checks passed" if all_passed else "\nSome checks failed")

## Step 9. Clean up

In [ ]:
conn.close()
print("✓ Database connection closed")

## Complete Script

The full recipe as one script. Start Ollama and pull `granite4.1:3b` first.

~~~~python
import re
import sqlite3
from typing import Literal

import pandas as pd
import pydantic
import sqlparse
from mellea import generative, start_session
from mellea.stdlib.requirement import req, simple_validate
from mellea.stdlib.sampling import RepairTemplateStrategy

conn = sqlite3.connect(":memory:", check_same_thread=False)

SCHEMA_DDL = """
CREATE TABLE player_stats (
    player_id INTEGER PRIMARY KEY,
    player_name TEXT NOT NULL,
    year INTEGER,
    total_matches INTEGER,
    wins INTEGER,
    losses INTEGER,
    win_percentage REAL
);
CREATE TABLE match_details (
    detail_id INTEGER PRIMARY KEY,
    player_id INTEGER REFERENCES player_stats(player_id),
    fastest_delivery_speed REAL,
    most_points_won_against TEXT,
    most_points_lost_against TEXT
);
"""
conn.executescript(SCHEMA_DDL)

player_stats = [
    (1, "Jasprit Bumrah", 2023, 28, 19, 9, 67.9),
    (2, "Jasprit Bumrah", 2022, 15, 10, 5, 66.7),
    (3, "Jasprit Bumrah", 2021, 24, 16, 8, 66.7),
    (4, "Mohammed Shami", 2023, 22, 13, 9, 59.1),
    (5, "Mohammed Shami", 2022, 18, 10, 8, 55.6),
    (6, "Ravindra Jadeja", 2023, 30, 20, 10, 66.7),
]
match_details = [
    (1, 1, 153.0, "Australia", "England"),
    (2, 2, 150.6, "South Africa", "Australia"),
    (3, 3, 152.1, "England", "Pakistan"),
    (4, 4, 147.8, "Pakistan", "India"),
    (5, 5, 146.2, "India", "Australia"),
    (6, 6, 121.5, "New Zealand", "England"),
]
conn.executemany("INSERT INTO player_stats VALUES (?, ?, ?, ?, ?, ?, ?)", player_stats)
conn.executemany("INSERT INTO match_details VALUES (?, ?, ?, ?, ?)", match_details)
conn.commit()

display(pd.read_sql_query("SELECT * FROM player_stats", conn))

ORIGINAL_QUERY = """
SELECT
    player_name,
    total_matches,
    wins,
    losses,
    win_percentage
FROM player_stats
WHERE player_name = 'Jasprit Bumrah'
ORDER BY year DESC;
"""

df_original = pd.read_sql_query(ORIGINAL_QUERY, conn)
display(df_original)

REQUIRED_COLUMNS = ["player_name", "total_matches", "wins", "losses", "win_percentage"]
NEW_COLUMNS = ["fastest_delivery_speed", "most_points_won_against", "most_points_lost_against"]


def clean_sql(text: str) -> str:
    """Strip Markdown code fences that models often wrap around SQL."""
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    return (match.group(1) if match else text).strip()


class HardcodedParameterCheck(pydantic.BaseModel):
    contains_hardcoded: Literal["yes", "no"]
    hardcoded_value: str | None = None  # The hardcoded value found, if any


def validate_sql_syntax(sql: str) -> bool:
    """Check that the query is a single SELECT and that SQLite can plan it against the real schema."""
    sql = clean_sql(sql)
    selects = [s for s in sqlparse.parse(sql) if s.get_type() == "SELECT"]
    if len(selects) != 1:
        print("SQL validation failed: expected exactly one SELECT statement")
        return False
    # Give every named parameter a placeholder value so EXPLAIN can run without executing the query
    params = {name: "placeholder" for name in re.findall(r":(\w+)", sql)}
    try:
        conn.execute(f"EXPLAIN QUERY PLAN {sql}", params)
        return True
    except sqlite3.Error as e:
        print(f"SQL validation failed: {e}")
        return False


def validate_backward_compatible(sql: str) -> bool:
    """Check that the new query still returns every original column."""
    sql = clean_sql(sql)
    return all(re.search(rf"\b{col}\b", sql, re.IGNORECASE) for col in REQUIRED_COLUMNS)


def validate_uses_named_parameter(sql: str) -> bool:
    """Check for the :player_name parameter and no quoted string literals."""
    sql = clean_sql(sql)
    return ":player_name" in sql and re.search(r"'[^']*'", sql) is None


@generative
def contains_hardcoded_parameters(sql_query: str) -> HardcodedParameterCheck:
    """Determines if the SQL query contains hardcoded values for parameters such as player names, player ids, dates etc. Instead of specific names, id or dates, the SQL query must use variables such as player_name, player_id, date etc.

    Returns:
        HardcodedParameterCheck with:
        - contains_hardcoded: 'yes' if hardcoded parameters found, 'no' otherwise
        - hardcoded_value: The actual hardcoded value found (e.g., 'Jasprit Bumrah'), or None if no hardcoded values
    """

m = start_session(backend_name="ollama", model_id="granite4.1:3b")


@generative
def needs_modernization_or_contains_hardcoded_parameter(sql_query: str) -> Literal["yes", "no"]:
    """Determines if the SQL query is outdated or has hardcoded parameter. If it has hardcoded parameter it needs modernization. Parameters can be things like player names, player ids, dates etc."""


needs_update = needs_modernization_or_contains_hardcoded_parameter(m, sql_query=ORIGINAL_QUERY)
print(f"Needs modernization? {needs_update}")

def validate_no_hardcoded_params(sql: str) -> bool:
    """Ask the generative check whether the query hardcodes any value."""
    result = contains_hardcoded_parameters(m, sql_query=clean_sql(sql))
    if result.contains_hardcoded == "yes":
        print(f"⚠️  Hardcoded parameter detected: {result.hardcoded_value}")
        return False
    return True


MODERNIZE_INSTRUCTION = """Modernize the SQL query below:
- Add fastest_delivery_speed, most_points_won_against, and most_points_lost_against from the match_details table.
- Join match_details to player_stats on player_id.
- Replace the hardcoded player name with the :player_name parameter.
- Keep every original column.

Return ONLY the SQL query."""

if needs_update == "yes":
    result = m.instruct(
        MODERNIZE_INSTRUCTION,
        grounding_context={"original_query": ORIGINAL_QUERY, "schema": SCHEMA_DDL},
        requirements=[
            req("SQL must be a single valid SELECT statement", validation_fn=simple_validate(validate_sql_syntax)),
            req("SQL must include all original columns", validation_fn=simple_validate(validate_backward_compatible)),
            req("SQL must use the :player_name parameter and no string literals", validation_fn=simple_validate(validate_uses_named_parameter)),
            req("SQL must not hardcode any value (checked with @generative)", validation_fn=simple_validate(validate_no_hardcoded_params)),
        ],
        strategy=RepairTemplateStrategy(loop_budget=5),
        return_sampling_results=True,
    )

    if result.success:
        modernized_query = clean_sql(str(result.result))
        print(f"✓ Passed all requirements after {len(result.sample_generations)} attempt(s)")
        print(f"\nModernized query:\n{modernized_query}")
    else:
        modernized_query = None
        print(f"✗ No passing query after {len(result.sample_generations)} attempt(s)")
else:
    modernized_query = None
    print("Query is already modern")

PLAYERS = ["Jasprit Bumrah", "Mohammed Shami", "Ravindra Jadeja"]

if modernized_query:
    frames = []
    for name in PLAYERS:
        frame = pd.read_sql_query(modernized_query, conn, params={"player_name": name})
        frames.append(frame)
    modern_df = pd.concat(frames, ignore_index=True)
    display(modern_df)
else:
    modern_df = None
    print("No modernized query to execute")

CORE_COLUMNS = ["player_name", "total_matches", "wins", "losses", "win_percentage"]

if modern_df is None:
    print("Cannot validate: no modernized query was produced")
else:
    all_passed = True
    for name in PLAYERS:
        expected = pd.read_sql_query(
            "SELECT player_name, total_matches, wins, losses, win_percentage FROM player_stats WHERE player_name = ?",
            conn,
            params=(name,),
        )
        actual = modern_df.loc[modern_df["player_name"] == name, CORE_COLUMNS]
        key = list(expected.columns)
        try:
            pd.testing.assert_frame_equal(
                expected.sort_values(key).reset_index(drop=True),
                actual.sort_values(key).reset_index(drop=True),
            )
            print(f"✓ {name}: {len(expected)} row(s) match ground truth")
        except AssertionError as e:
            all_passed = False
            print(f"✗ {name}: {e}")

    new_cols_filled = modern_df[NEW_COLUMNS].notna().all().all()
    print(f"{'✓' if new_cols_filled else '✗'} New columns are populated")
    all_passed = all_passed and new_cols_filled
    print("\nAll checks passed" if all_passed else "\nSome checks failed")

conn.close()
print("✓ Database connection closed")
~~~~

## Running the Notebook

Start Ollama and pull the model, then run all cells top to bottom. Set `start_session` to a different backend if you prefer watsonx.ai.

## Key Takeaways

- Instruct-Validate-Repair turns a one-off SQL rewrite into a checked process: each requirement tests one property.
- Deterministic checks (syntax, required columns, named parameters) are fast and repeatable. A `@generative` check covers judgments that code can't make.
- `EXPLAIN QUERY PLAN` validates a query against the real schema without running it.
- Named parameters let one query serve every player with no string substitution.
- Ground-truth comparison confirms the rewritten query returns the same data as the source table.

## Next Steps

- [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb): the core pattern used here.
- [Support Ticket Triage](../SupportTicketTriage/SupportTicketTriage.ipynb): applies typed handoffs and tools to a multi-agent pipeline.